In [3]:
// ── Step 1: CRC32テーブルの構築 ──────────────────────────────────────────────
fn build_crc32_table() -> [u32; 256] {
    let mut table = [0u32; 256];
    for i in 0..256u32 {
        let mut c = i;
        for _ in 0..8 {
            if c & 1 != 0 { c = 0xEDB8_8320 ^ (c >> 1); } else { c >>= 1; }
        }
        table[i as usize] = c;
    }
    table
}

fn crc32_update(table: &[u32; 256], crc: u32, b: u8) -> u32 {
    (crc >> 8) ^ table[((crc ^ b as u32) & 0xFF) as usize]
}

let table = build_crc32_table();
for i in 0..256{ println!("  table[{}] = 0x{:08x}", i, table[i]); }
println!("✓ CRC32テーブル構築完了 (256エントリ)");

CRC32テーブル先頭4エントリ:
  table[0] = 0x00000000
  table[1] = 0x77073096
  table[2] = 0xee0e612c
  table[3] = 0x990951ba
  table[4] = 0x076dc419
  table[5] = 0x706af48f
  table[6] = 0xe963a535
  table[7] = 0x9e6495a3
  table[8] = 0x0edb8832
  table[9] = 0x79dcb8a4
  table[10] = 0xe0d5e91e
  table[11] = 0x97d2d988
  table[12] = 0x09b64c2b
  table[13] = 0x7eb17cbd
  table[14] = 0xe7b82d07
  table[15] = 0x90bf1d91
  table[16] = 0x1db71064
  table[17] = 0x6ab020f2
  table[18] = 0xf3b97148
  table[19] = 0x84be41de
  table[20] = 0x1adad47d
  table[21] = 0x6ddde4eb
  table[22] = 0xf4d4b551
  table[23] = 0x83d385c7
  table[24] = 0x136c9856
  table[25] = 0x646ba8c0
  table[26] = 0xfd62f97a
  table[27] = 0x8a65c9ec
  table[28] = 0x14015c4f
  table[29] = 0x63066cd9
  table[30] = 0xfa0f3d63
  table[31] = 0x8d080df5
  table[32] = 0x3b6e20c8
  table[33] = 0x4c69105e
  table[34] = 0xd56041e4
  table[35] = 0xa2677172
  table[36] = 0x3c03e4d1
  table[37] = 0x4b04d447
  table[38] = 0xd20d85fd
  table[39] = 0xa

In [4]:
// ── Step 2: PkzipKeys 構造体 ─────────────────────────────────────────────────
#[derive(Clone, Debug)]
struct PkzipKeys { k0: u32, k1: u32, k2: u32 }

impl PkzipKeys {
    /// パスワードから内部鍵を初期化
    fn from_password(table: &[u32; 256], password: &[u8]) -> Self {
        let mut keys = PkzipKeys { k0: 0x12345678, k1: 0x23456789, k2: 0x34567890 };
        for &b in password { keys.update(table, b); }
        keys
    }

    /// 1バイト処理するたびに鍵状態を更新
    fn update(&mut self, table: &[u32; 256], b: u8) {
        self.k0 = crc32_update(table, self.k0, b);
        self.k1 = self.k1.wrapping_add(self.k0 & 0xFF);
        self.k1 = self.k1.wrapping_mul(0x0808_8405).wrapping_add(1);
        self.k2 = crc32_update(table, self.k2, (self.k1 >> 24) as u8);
    }

    /// キーストリームバイトを生成
    fn stream_byte(&self) -> u8 {
        let t = (self.k2 | 2) as u16;
        ((t.wrapping_mul(t ^ 1)) >> 8) as u8
    }

    /// 復号: cipher_byte XOR keystream
    fn decrypt_byte(&mut self, table: &[u32; 256], cipher_byte: u8) -> u8 {
        let plain = cipher_byte ^ self.stream_byte();
        self.update(table, plain);
        plain
    }

    /// 暗号化: plain_byte XOR keystream
    fn encrypt_byte(&mut self, table: &[u32; 256], plain: u8) -> u8 {
        let c = plain ^ self.stream_byte();
        self.update(table, plain);
        c
    }
}

println!("✓ PkzipKeys 定義完了");

✓ PkzipKeys 定義完了


In [5]:
// ── Step 3: 動作確認 ─────────────────────────────────────────────────────────
let password = b"test";
let plaintext = b"FLAG_TestFlag1234";

// 暗号化
let mut enc_keys = PkzipKeys::from_password(&table, password);
let ciphertext: Vec<u8> = plaintext.iter().map(|&b| enc_keys.encrypt_byte(&table, b)).collect();
println!("平文:   {:?}", String::from_utf8_lossy(plaintext));
println!("暗号文: {:02x?}", &ciphertext);

// 復号
let mut dec_keys = PkzipKeys::from_password(&table, password);
let decrypted: Vec<u8> = ciphertext.iter().map(|&cb| dec_keys.decrypt_byte(&table, cb)).collect();
println!("復号:   {:?}", String::from_utf8_lossy(&decrypted));

assert_eq!(plaintext.as_slice(), decrypted.as_slice());
println!("✓ 暗号化/復号 正常動作確認");

平文:   "FLAG_TestFlag1234"
暗号文: [99, 50, 27, 87, c7, fe, 07, 9e, 45, ed, d0, 8a, 09, d4, 40, e8, 9b]
復号:   "FLAG_TestFlag1234"
✓ 暗号化/復号 正常動作確認


In [6]:
// ── Step 4: ZIPローカルヘッダのパース ────────────────────────────────────────
#[derive(Debug)]
struct LocalFileEntry {
    filename: String,
    flags: u16,
    compressed_data: Vec<u8>,
}

fn read_zip_local_entries(data: &[u8]) -> Vec<LocalFileEntry> {
    let mut entries = Vec::new();
    let mut pos = 0;
    while pos + 30 <= data.len() {
        // ローカルファイルヘッダのシグネチャ: PK\x03\x04
        let sig = u32::from_le_bytes(data[pos..pos+4].try_into().unwrap());
        if sig != 0x04034B50 { break; }
        let flags     = u16::from_le_bytes(data[pos+6..pos+8].try_into().unwrap());
        let comp_size = u32::from_le_bytes(data[pos+18..pos+22].try_into().unwrap()) as usize;
        let fname_len = u16::from_le_bytes(data[pos+26..pos+28].try_into().unwrap()) as usize;
        let extra_len = u16::from_le_bytes(data[pos+28..pos+30].try_into().unwrap()) as usize;
        let fname_start = pos + 30;
        let fname_end   = fname_start + fname_len;
        let data_start  = fname_end + extra_len;
        let data_end    = data_start + comp_size;
        let filename = String::from_utf8_lossy(&data[fname_start..fname_end]).to_string();
        let compressed_data = data[data_start..data_end].to_vec();
        entries.push(LocalFileEntry { filename, flags, compressed_data });
        pos = data_end;
    }
    entries
}

println!("✓ ZIPパーサー定義完了");
println!();
println!("ZIPローカルファイルヘッダ構造:");
println!("  Offset  Size  Description");
println!("  0       4     Signature (PK\\x03\\x04)");
println!("  4       2     Version needed");
println!("  6       2     General purpose bit flag (bit0=1: 暗号化)");
println!("  8       2     Compression method");
println!("  18      4     Compressed size");
println!("  26      2     File name length");
println!("  28      2     Extra field length");
println!("  30      n     File name");
println!("  30+n    m     Extra field");
println!("  30+n+m  ...   File data (暗号化: 12バイトヘッダ + 本体)");

✓ ZIPパーサー定義完了

ZIPローカルファイルヘッダ構造:
  Offset  Size  Description
  0       4     Signature (PK\x03\x04)
  4       2     Version needed
  6       2     General purpose bit flag (bit0=1: 暗号化)
  8       2     Compression method
  18      4     Compressed size
  26      2     File name length
  28      2     Extra field length
  30      n     File name
  30+n    m     Extra field
  30+n+m  ...   File data (暗号化: 12バイトヘッダ + 本体)


In [7]:
// ── Step 5: ブルートフォース (短いパスワードに対して) ────────────────────────
fn bruteforce_password(
    table: &[u32; 256],
    entry: &LocalFileEntry,
    check_magic: &[u8],
) -> Option<String> {
    let ciphertext = &entry.compressed_data;
    if ciphertext.len() < 12 { return None; }
    let charset: Vec<u8> = (b'a'..=b'z').chain(b'A'..=b'Z').chain(b'0'..=b'9').collect();
    for len in 1..=4usize {
        let count = charset.len().pow(len as u32);
        println!("  [*] 長さ {} の候補 {} 件を試行中...", len, count);
        for idx in 0..count {
            let mut pwd = Vec::with_capacity(len);
            let mut n = idx;
            for _ in 0..len { pwd.push(charset[n % charset.len()]); n /= charset.len(); }
            let mut keys = PkzipKeys::from_password(table, &pwd);
            let plainbuf: Vec<u8> = ciphertext.iter().map(|&cb| keys.decrypt_byte(table, cb)).collect();
            if plainbuf.len() >= 12 + check_magic.len() {
                if &plainbuf[12..12 + check_magic.len()] == check_magic {
                    return Some(String::from_utf8_lossy(&pwd).to_string());
                }
            }
        }
    }
    None
}

fn decrypt_entry(table: &[u32; 256], entry: &LocalFileEntry, password: &str) -> Vec<u8> {
    let mut keys = PkzipKeys::from_password(table, password.as_bytes());
    let mut buf: Vec<u8> = entry.compressed_data.iter().map(|&cb| keys.decrypt_byte(table, cb)).collect();
    buf[12..].to_vec()  // 先頭12バイト(ランダムヘッダ)を除く
}

println!("✓ ブルートフォース関数定義完了");

✓ ブルートフォース関数定義完了


In [8]:
// ── Step 6: 既知平文攻撃のシミュレーション ───────────────────────────────────
// 実際のflag.zipがない環境でも動作確認できるデモ

println!("=== 既知平文攻撃シミュレーション ===");

// ① シークレットパスワードでJPEGを暗号化したZIPを模倣
let secret_pwd = b"abc";  // 攻撃者は知らないパスワード

// JPEGマジックバイト FF D8 FF E0 を含む既知平文
let known_plain: &[u8] = b"\xff\xd8\xff\xe0This is Standard-lock-key.jpg file content...";

// 12バイトランダムヘッダ (実際は乱数; ここでは固定値でデモ)
let rand_header: Vec<u8> = (0u8..12).collect();

let mut enc = PkzipKeys::from_password(&table, secret_pwd);
let mut cipher_data = Vec::new();
for &b in &rand_header    { cipher_data.push(enc.encrypt_byte(&table, b)); }
for &b in known_plain     { cipher_data.push(enc.encrypt_byte(&table, b)); }

println!("[+] 秘密パスワード: {:?} でJPEGを暗号化", String::from_utf8_lossy(secret_pwd));
println!("[+] 暗号化データ先頭: {:02x?}", &cipher_data[..16]);

// ② 攻撃: JPEGマジックバイト(FF D8 FF)を使って総当たり
let fake_entry = LocalFileEntry {
    filename: "Standard-lock-key.jpg".to_string(),
    flags: 1,
    compressed_data: cipher_data,
};

let jpeg_magic: &[u8] = &[0xFF, 0xD8, 0xFF];
// 小文字のみで高速デモ
let charset: Vec<u8> = (b'a'..=b'z').collect();
let mut found_pwd = None;
'outer: for len in 1..=3usize {
    for idx in 0..charset.len().pow(len as u32) {
        let mut pwd = Vec::new();
        let mut n = idx;
        for _ in 0..len { pwd.push(charset[n % charset.len()]); n /= charset.len(); }
        let mut keys = PkzipKeys::from_password(&table, &pwd);
        let plain: Vec<u8> = fake_entry.compressed_data.iter().map(|&cb| keys.decrypt_byte(&table, cb)).collect();
        if plain.len() >= 15 && &plain[12..15] == jpeg_magic {
            found_pwd = Some(String::from_utf8_lossy(&pwd).to_string());
            break 'outer;
        }
    }
}

match found_pwd {
    Some(p) => println!("\n🎉 既知平文攻撃成功! パスワード: {:?}", p),
    None    => println!("[!] 見つかりませんでした"),
};

=== 既知平文攻撃シミュレーション ===
[+] 秘密パスワード: "abc" でJPEGを暗号化
[+] 暗号化データ先頭: [1f, 31, 08, f9, 2e, d6, 80, b3, 91, 04, b7, 48, 64, 2a, f4, d0]

🎉 既知平文攻撃成功! パスワード: "abc"


In [2]:
use std::process::Command;

{
    let output = Command::new("bkcrack")
        .arg("--version")
        .output();

    match output {
        Ok(res) => {
            println!("✅ bkcrack の呼び出しに成功しました！");
            println!("{}", String::from_utf8_lossy(&res.stdout));
        }
        Err(_) => {
            println!("❌ bkcrack が見つかりません。パスが通っているか、フルパスで指定してください。");
        }
    }
}

✅ bkcrack の呼び出しに成功しました！
bkcrack 1.8.1 - 2025-10-25



()

In [6]:
use std::process::Command;

let output = Command::new("bkcrack")
    .arg("-L")
    .arg("flag.zip")
    .output();

match output {
    Ok(res) => {
        println!("--- stdout ---\n{}", String::from_utf8_lossy(&res.stdout));
        println!("--- stderr ---\n{}", String::from_utf8_lossy(&res.stderr));
    }
    Err(e) => println!("実行失敗: {}", e),
}

--- stdout ---
bkcrack 1.8.1 - 2025-10-25
Archive: flag.zip
Index Encryption Compression CRC32    Uncompressed  Packed size Name
----- ---------- ----------- -------- ------------ ------------ ----------------
    0 ZipCrypto  Store       d66cb67b          304          316 flag.html
    1 ZipCrypto  Store       af308dc8       255964       255976 Standard-lock-key.jpg

--- stderr ---



()

In [7]:
use std::process::Command;

let output = Command::new("bkcrack")
    .arg("-C").arg("flag.zip")
    .arg("-c").arg("Standard-lock-key.jpg") // ZIP内のファイル名
    .arg("-p").arg("Standard-lock-key.jpg") // ローカルにある平文ファイル
    .arg("-D").arg("decrypted.zip")         // 成功時の出力先
    .output();

match output {
    Ok(res) => {
        let stdout = String::from_utf8_lossy(&res.stdout);
        let stderr = String::from_utf8_lossy(&res.stderr);
        println!("--- stdout ---\n{}", stdout);
        println!("--- stderr ---\n{}", stderr);
        
        if res.status.success() {
            println!("🎉 成功しました！ 'decrypted.zip' が生成されたはずです。");
        } else {
            println!("❌ bkcrackがエラーを返しました。stderrを確認してください。");
        }
    }
    Err(e) => println!("実行失敗: {}", e),
}

--- stdout ---
bkcrack 1.8.1 - 2025-10-25
[17:03:47] Z reduction using 255957 bytes of known plaintext
5.4 % (13744 / 255957)
[17:03:48] Attack on 105 Z values at index 242638
Keys: 7adffffe 468d5ff6 259a116a
54.3 % (57 / 105)
Found a solution. Stopping.
You may resume the attack with the option: --continue-attack 57
[17:03:48] Keys
7adffffe 468d5ff6 259a116a
[17:03:48] Writing decrypted archive decrypted.zip
100.0 % (2 / 2)

--- stderr ---

🎉 成功しました！ 'decrypted.zip' が生成されたはずです。


()